# Quantvesting

## Premium Investment Terminal — Phase 2

A notebook-first executive view over the existing Quantvesting engine. **No investment calculations live in the notebook.**

In [ ]:
!pip install -q ta pyxirr matplotlib plotly ipywidgets

### 1. Environment & controls

Change only `PORTFOLIO_ID` for another portfolio. Keep `EOD_RUN=False` unless this is the final end-of-day snapshot.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
PROJECT = "/content/drive/My Drive/quantvesting_v3"
MARKET_DATA_DIR = PROJECT + "/market_data"
PORTFOLIO_ID = "ankit"
PORTFOLIO_DATA_DIR = PROJECT + f"/portfolio_data/{PORTFOLIO_ID}"
EOD_RUN = False

sys.path.insert(0, PROJECT + "/src")

from quantvesting import (
    Quantvesting, load_config, load_market_data, load_portfolio_data,
    create_run_id, PROSPECT_DISPLAY_COLUMNS, PORTFOLIO_DISPLAY_COLUMNS,
)

RUN_ID = create_run_id()

In [ ]:
config = load_config(PROJECT + "/config/strategy.yaml")
qv = Quantvesting(config)

# Optional shared-market refresh; leave False unless a new Screener XLSX arrived.
REFRESH_SCREENER = False
if REFRESH_SCREENER:
    qv.ingest_screener(MARKET_DATA_DIR)

market_data = load_market_data(MARKET_DATA_DIR)
portfolio_data = load_portfolio_data(PORTFOLIO_DATA_DIR, portfolio_id=PORTFOLIO_ID)

### 2. Run the existing engine

The notebook only orchestrates the existing prospect/portfolio/decision layers.

In [ ]:
df_prospects = qv.prospects(
    market_data, portfolio_data=portfolio_data, include_portfolio=True,
    portfolio_id=PORTFOLIO_ID, run_id=RUN_ID,
)

df_portfolio, portfolio_summary = qv.portfolio(
    market_data, portfolio_data=portfolio_data, eod=EOD_RUN,
    portfolio_id=PORTFOLIO_ID, run_id=RUN_ID,
)

df_prospect_actions = qv.prospect_actions(df_prospects, top_n=10)
df_portfolio_actions = qv.portfolio_actions(df_portfolio)
df_rotation = qv.capital_rotation(df_prospects, df_portfolio)

### 3. Executive dashboard

The top section is designed to answer four questions quickly: **What do I own? What needs attention? Where is the remaining upside? Where is the next opportunity?**

In [ ]:
qv.display_terminal(
    df_portfolio=df_portfolio,
    df_prospects=df_prospects,
    portfolio_summary=portfolio_summary,
    df_rotation=df_rotation,
    df_portfolio_actions=df_portfolio_actions,
    df_prospect_actions=df_prospect_actions,
    portfolio_id=PORTFOLIO_ID,
    run_id=RUN_ID,
    run_datetime=portfolio_summary.get("run_datetime"),
)

### 4. Interactive portfolio intelligence

Use the tables for drill-down. Sorting/filtering remains delegated to the existing Jupyter/Colab table adapter.

In [ ]:
qv.display_dataframe(
    df_portfolio_actions,
    columns=[c for c in PORTFOLIO_DISPLAY_COLUMNS + ["Action", "ActionReason", "ActionEvidence"]
             if c in df_portfolio_actions.columns],
    sort_by="CurrAlloc%", ascending=False,
)

In [ ]:
qv.display_dataframe(
    df_prospect_actions,
    columns=[c for c in PROSPECT_DISPLAY_COLUMNS + ["Action", "Reason", "Evidence"]
             if c in df_prospect_actions.columns],
    sort_by="CumlRnk", ascending=True,
)

### 5. Visual views

In [ ]:
qv.display_health_chart(df_portfolio)
qv.display_upside_chart(df_portfolio, top_n=12)
qv.display_prospect_opportunities(df_prospects, top_n=12)

### 6. Capital rotation review

This is advisory only. It does not issue an automatic sell instruction.

In [ ]:
if df_rotation.empty:
    print("No capital-rotation candidates at the current configured thresholds.")
else:
    display(df_rotation)

### 7. Run / date selector

Run manifests answer **which data/configuration produced a result**. EOD history provides the stored portfolio time series. Historical per-stock snapshots are not reconstructed by this selector because the current repository does not persist those per-stock snapshots.

In [ ]:
qv.display_run_history_selector(portfolio_data, current_run_id=RUN_ID)

### 8. HNI review checklist

1. Review the executive cards.
2. Read active actions and their evidence.
3. Inspect top remaining-upside holdings.
4. Inspect top prospect opportunities.
5. Review rotation candidates.
6. Check the selected run/date before sharing the report.

The investment engine remains unchanged; this notebook is the presentation layer.